# VLM Visual Math Reasoning Walkthrough

This notebook mirrors the homework workflow from the vision-language-model chapter. It keeps the answer contract, prompt-only baseline, parser tests, adaptation or fallback comparison, and final error taxonomy in one inspectable sequence.

The cells use a tiny generated visual-math sample by default, so readers can understand the scoring harness without downloading a VLM. Real model generation should happen in a reviewed GPU environment, then the saved JSONL outputs can be scored with the same parser and summary functions shown here.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

## 1. Import The Scoring Helpers

This setup cell locates the chapter directory, imports the parser and evaluation helpers, and chooses a local ignored data directory for sample images. It should not contact a model provider or read secrets; it only prepares the notebook to generate sample records and score saved text completions.

In [ ]:
from pathlib import Path
import json
import sys


def find_chapter_dir(script_name: str, chapter_name: str) -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / chapter_name, base / "code" / chapter_name):
            if (candidate / script_name).exists():
                return candidate.resolve()
    raise FileNotFoundError(f"Could not locate {script_name}")


CHAPTER_DIR = find_chapter_dir("vlm_eval.py", "chapter_vision_language_models")
sys.path.insert(0, str(CHAPTER_DIR))

from answer_parser import parse_answer, score_completion, unit_test_parser
from vlm_eval import (
    ANSWER_CONTRACT,
    load_records,
    prompt_for_question,
    score_generated_rows,
    summarize_scores,
    write_sample_data,
)

DATA_DIR = CHAPTER_DIR / "data" / "sample_visual_math"
PROMPTS_PATH = DATA_DIR / "prompts.jsonl"

## 2. Freeze The Answer Contract

The scoring contract requires exactly one non-empty `<answer>...</answer>` tag. Printing the contract and running parser unit tests before generation makes the expected output format explicit, which prevents a model comparison from mixing reasoning quality with avoidable parsing mistakes.

In [ ]:
contract = {
    "answer_format": "exactly one <answer>...</answer> tag",
    "exact_match_rule": "normalize case, whitespace, and simple decimal variants",
    "valid_format_rule": "one non-empty answer tag",
    "prompt_template": ANSWER_CONTRACT,
}
print(json.dumps(contract, indent=2))
unit_test_parser()

## 3. Create The Sample Visual-Math Data

This cell writes a tiny PNG dataset and matching prompt records under `data/sample_visual_math/`. The directory is ignored by git, and the records are intentionally small so readers can inspect IDs, splits, image paths, questions, and reference answers before running or pasting model outputs.

In [ ]:
write_sample_data(DATA_DIR)
records = load_records(PROMPTS_PATH)
records[:2]

## 4. Preview Prompts Before Running A Model

The preview prints the rendered prompt and reference answer for the first records. In a GPU environment, run the matching command-line generation step shown in the comment, save outputs as JSONL, and keep model name, revision, runtime package versions, and hardware metadata with the run.

In [ ]:
for record in records[:2]:
    print(record["id"])
    print(record["image_path"])
    print(prompt_for_question(record["question"]))
    print("reference:", record["reference_answer"])
    print()

# In a GPU environment, run the matching command-line baseline:
# python vlm_eval.py --generate --prompts data/sample_visual_math/prompts.jsonl

## 5. Unit-Test The Parser And Reward

These examples cover valid format, missing tags, wrong answers, and multiple tags. Check them before scoring a real run so exact-match accuracy and valid-format rate mean the same thing across the baseline and any adaptation or fallback ablation.

In [ ]:
examples = [
    ("<answer>12</answer>", "12"),
    ("The answer is 12.", "12"),
    ("<answer>11</answer>", "12"),
    ("<answer>12</answer><answer>12</answer>", "12"),
]

for completion, reference in examples:
    score = score_completion(completion, reference)
    print(completion, score.to_dict())

## 6. Compare A Baseline With One Ablation

The synthetic rows stand in for generated model outputs and exercise the same scoring path a real JSONL file would use. Keep the prompt set and final-test split fixed while changing only one condition, such as image resolution, prompt wording, model checkpoint, or adaptation method.

In [ ]:
synthetic_baseline_outputs = []
synthetic_ablation_outputs = []
for record in records:
    baseline_text = f"<answer>{record['reference_answer']}</answer>" if record["id"] != "chart_001" else "<answer>A</answer>"
    ablation_text = f"<answer>{record['reference_answer']}</answer>"
    synthetic_baseline_outputs.append({**record, "run": "baseline", "generated_text": baseline_text})
    synthetic_ablation_outputs.append({**record, "run": "larger_image_ablation", "generated_text": ablation_text})

baseline_scored = score_generated_rows(synthetic_baseline_outputs)
ablation_scored = score_generated_rows(synthetic_ablation_outputs)
print("baseline", summarize_scores(baseline_scored))
print("ablation", summarize_scores(ablation_scored))

## 7. Build The Final Error Taxonomy

After selecting a model using validation evidence, summarize the held-out `final_test` rows once and label the remaining errors. Use categories such as perception, counting, chart reading, arithmetic, instruction following, or invalid format, and cite representative examples without exposing private or licensed image content.

In [ ]:
final_rows = [row for row in baseline_scored if row["split"] == "final_test"]
for row in final_rows:
    row["error_category"] = "perception" if not row["exact_match"] else "none"

summary = summarize_scores(final_rows)
print(json.dumps(summary, indent=2, sort_keys=True))
final_rows

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.